# Trial — V1 + V2 Only (2 leads)
**Brugada-HUCA ECG Dataset**

This is a reduced-lead trial using only **V1 and V2**, the two leads most clinically relevant for Brugada syndrome (coved ST pattern).

Pipeline:
1. Load raw 12-lead signals
2. Select leads V1 (index 6) and V2 (index 7)
3. Filter → normalize → split
4. Train 1D ResNet with `n_leads=2`
5. Compare vs 12-lead baseline

## 0 — Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'src' / 'config.py').exists()),
    _cwd,
)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import config as cfg
from preprocessing import (
    load_all_records, preprocess_batch,
    make_splits, compute_pos_weight, save_processed, load_processed,
)
from dataset import ECGDataset, make_loaders
from models import ResNet1D
from train import run_training, load_checkpoint
from evaluate import compute_metrics, predict_dl, plot_training_curves

# --- Trial-specific config ---
TRIAL_NAME    = 'v1v2'
SELECTED_LEADS = [6, 7]          # V1=6, V2=7
LEAD_NAMES     = ['V1', 'V2']
N_LEADS        = len(SELECTED_LEADS)   # 2

PROCESSED_DIR  = ROOT / 'data' / f'processed_{TRIAL_NAME}'
MODELS_DIR     = ROOT / 'models' / TRIAL_NAME
REPORTS_DIR    = ROOT / 'reports' / TRIAL_NAME
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

print(f'Trial       : {TRIAL_NAME}')
print(f'Leads       : {LEAD_NAMES}  (indices {SELECTED_LEADS})')
print(f'Device      : {DEVICE}')
print(f'Processed   : {PROCESSED_DIR}')
print(f'Models      : {MODELS_DIR}')

## 1 — Load Raw Signals & Select Leads

In [ ]:
meta = pd.read_csv(cfg.METADATA_FILE)

print('Loading all records (12-lead)...')
X_raw_12, y, ids = load_all_records(meta, cfg.DATA_RAW, verbose=True)
print(f'\nFull shape  : {X_raw_12.shape}  (N, 12 leads, samples)')

# --- Select V1 and V2 only ---
X_raw = X_raw_12[:, SELECTED_LEADS, :]   # (N, 2, 1200)
print(f'After slice : {X_raw.shape}  (N, {N_LEADS} leads, samples)')
print(f'Labels      : Brugada={(y==1).sum()}  Normal={(y==0).sum()}')

## 2 — Preprocess (Filter + Normalize)

In [ ]:
print('Applying bandpass → notch → Z-score normalization...')
X_clean = preprocess_batch(X_raw)   # (N, 2, 1200)

print(f'X_clean shape : {X_clean.shape}')
print(f'  mean  = {X_clean.mean():.6f}  (expect ~0)')
print(f'  std   = {X_clean.std():.6f}   (expect ~1)')

In [ ]:
# Visualise: one Brugada patient — V1 and V2 after preprocessing
b_idx = np.where(y == 1)[0][0]
time  = np.arange(cfg.N_SAMPLES) / cfg.FS

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
fig.suptitle(f'Preprocessed Signal — V1 & V2 (Brugada ID: {ids[b_idx]})', fontsize=12, fontweight='bold')
for i, (ax, name) in enumerate(zip(axes, LEAD_NAMES)):
    ax.plot(time, X_clean[b_idx, i], linewidth=0.9)
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.4)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'preprocessed_signals.png', dpi=150)
plt.show()

## 3 — Train / Val / Test Split

In [ ]:
splits = make_splits(X_clean, y, ids)

print('Split sizes:')
for name in ('train', 'val', 'test'):
    y_s = splits[f'y_{name}']
    print(f'  {name:5s}: {len(y_s):3d} samples  |  '
          f'Brugada={(y_s==1).sum()}  Normal={(y_s==0).sum()}  '
          f'({100*(y_s==1).sum()/len(y_s):.1f}% positive)')

In [ ]:
# Save to trial-specific directory
print(f'Saving to {PROCESSED_DIR} ...')
from preprocessing import save_processed
save_processed(splits, PROCESSED_DIR)
print('Done.')

## 4 — Train 1D ResNet (n_leads=2)

In [ ]:
loaders = make_loaders(splits, batch_size=cfg.BATCH_SIZE)

model = ResNet1D(
    n_leads=N_LEADS,        # 2 instead of 12
    base_channels=64,
    kernel_size=7,
    dropout=0.2,
).to(DEVICE)

print(f'Parameters : {model.count_parameters():,}')

with torch.no_grad():
    dummy = torch.zeros(2, N_LEADS, cfg.N_SAMPLES).to(DEVICE)
    out   = model(dummy)
    print(f'Output shape: {out.shape}  (expected: [2, 1])')

In [ ]:
pos_w = compute_pos_weight(splits['y_train'])
print(f'pos_weight = {pos_w:.2f}')

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_w, dtype=torch.float32).to(DEVICE)
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-5
)

In [ ]:
CHECKPOINT = MODELS_DIR / 'best_resnet1d.pt'

history = run_training(
    model           = model,
    train_loader    = loaders['train'],
    val_loader      = loaders['val'],
    optimizer       = optimizer,
    criterion       = criterion,
    scheduler       = scheduler,
    device          = DEVICE,
    num_epochs      = cfg.NUM_EPOCHS,
    patience        = cfg.PATIENCE,
    checkpoint_path = CHECKPOINT,
    verbose         = True,
)

best_auroc = max(history['val_auroc'])
best_epoch = int(np.argmax(history['val_auroc'])) + 1
print(f'\nBest val AUROC: {best_auroc:.4f} at epoch {best_epoch}')

In [ ]:
plot_training_curves(history, save_path=REPORTS_DIR / 'training_curves.png')

## 5 — Validation & Test Evaluation

In [ ]:
best_epoch = load_checkpoint(CHECKPOINT, model)
print(f'Loaded best checkpoint from epoch {best_epoch}')

y_true_val,  y_prob_val  = predict_dl(model, loaders['val'],  device=DEVICE)
y_true_test, y_prob_test = predict_dl(model, loaders['test'], device=DEVICE)

metrics_val  = compute_metrics(y_true_val,  y_prob_val)
metrics_test = compute_metrics(y_true_test, y_prob_test)

print(f'\n--- Trial: {TRIAL_NAME} ({N_LEADS} leads: {LEAD_NAMES}) ---')
print('\nVal  metrics:')
for k, v in metrics_val.items():  print(f'  {k}: {v}')
print('\nTest metrics:')
for k, v in metrics_test.items(): print(f'  {k}: {v}')

In [ ]:
# Summary table
df_summary = pd.DataFrame({
    'Val':  metrics_val,
    'Test': metrics_test,
})
print(f'\nResNet1D — {TRIAL_NAME} ({N_LEADS}-lead) Summary:')
display(df_summary.round(4))

## Summary

| Setting | Value |
|---|---|
| Leads used | V1, V2 (2 of 12) |
| Lead indices | 6, 7 |
| Model | 1D ResNet, n_leads=2 |
| Preprocessing | Bandpass (0.5–40 Hz) → Notch (50 Hz) → Z-score per lead |
| Split | 70 / 15 / 15 %, stratified, seed=42 |
| Saved weights | `models/v1v2/best_resnet1d.pt` |

Compare results against the 12-lead baseline in `03_training.ipynb`.